# Yellow Trip Hourly Event-Table Preprocessing

이 노트북은 원본 `sample_data/new_york_taxi/yellow_trip.parquet`을 TPP 학습에 바로 사용할 수 있는 hourly marked-event 원천 테이블로 변환합니다.

기존 실험 코드 안에서 수행하던 내부 변환을 밖으로 분리하여, 학습 스크립트는 완성된 `yellow_trip_hourly.parquet`만 읽도록 만드는 것이 목적입니다.

## 1. 환경 설정

Jupyter kernel이 프로젝트 루트가 아닌 곳에서 시작해도 동작하도록 `PROJECT_ROOT`를 명시적으로 찾습니다.

In [ ]:
from pathlib import Path
import sys

import polars as pl
from IPython.display import display


def resolve_project_root(start: Path | None = None) -> Path:
    """Find the repository root from a notebook or script working directory."""
    anchor = (start or Path.cwd()).resolve()
    base = anchor if anchor.is_dir() else anchor.parent
    for candidate in [base, *base.parents]:
        if all((candidate / name).exists() for name in ("sample_data", "simple_lab_test", "utils")):
            return candidate
    fallback = Path("~/workspace/paper_research").expanduser().resolve()
    if all((fallback / name).exists() for name in ("sample_data", "simple_lab_test", "utils")):
        return fallback
    raise RuntimeError("Could not locate the paper_research project root.")


PROJECT_ROOT = resolve_project_root()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_PATH = PROJECT_ROOT / "sample_data" / "new_york_taxi" / "yellow_trip.parquet"
OUTPUT_PATH = PROJECT_ROOT / "sample_data" / "new_york_taxi" / "yellow_trip_hourly.parquet"

print("PROJECT_ROOT =", PROJECT_ROOT)
print("RAW_PATH     =", RAW_PATH)
print("OUTPUT_PATH  =", OUTPUT_PATH)

## 2. 변환 파라미터

`yellow_trip`은 개별 택시 승차 로그이므로, TPP 학습을 위해서는 시계열 단위가 필요합니다. 여기서는 pickup 위치를 격자 cell로 만들고, 각 cell의 시간별 pickup 수를 `demand_qty`로 사용합니다.

In [ ]:
# 1 hour bucket: event time is measured as elapsed hourly bucket count.
RESOLUTION = "hourly"
BUCKET_EVERY = "1h"

# Spatial bucket size. 0.02 degrees is the setting used by the long-epoch experiments.
GRID_SIZE_DEG = 0.02

# Keep only grid cells with enough positive hourly buckets to form meaningful sequences.
MIN_ACTIVE_BUCKETS = 72

# 0 means use all vendors; set 1 or 2 to reproduce vendor-specific subsets.
VENDOR_ID = 0

# Loose NYC-area bounds remove malformed coordinate rows without overfitting to a tiny zone.
MIN_LON, MAX_LON = -75.0, -72.0
MIN_LAT, MAX_LAT = 40.0, 42.0

# Set to an integer for a quick subset preprocessing run; keep None for the paper dataset.
MAX_SERIES = None

print({
    "resolution": RESOLUTION,
    "bucket_every": BUCKET_EVERY,
    "grid_size_deg": GRID_SIZE_DEG,
    "min_active_buckets": MIN_ACTIVE_BUCKETS,
    "vendor_id": VENDOR_ID,
    "max_series": MAX_SERIES,
})

## 3. 원본 데이터 확인

필수 컬럼은 pickup 시각, pickup 경도, pickup 위도입니다. 서버/로컬 parquet 스키마 차이가 있을 수 있어 timestamp는 문자열과 datetime을 모두 처리합니다.

In [ ]:
raw_df = pl.read_parquet(RAW_PATH)
print("raw rows =", raw_df.height)
print("raw columns =", raw_df.columns)
display(raw_df.head(5))

## 4. Hourly event table 생성 함수

출력 테이블의 핵심 컬럼은 아래와 같습니다.

- `oper_part_no`: pickup grid cell id. 기존 intermittent 코드의 series key와 맞추기 위해 같은 이름을 사용합니다.
- `demand_dt`: hourly bucket timestamp를 `yyyymmddHHMMSS` 정수로 저장합니다.
- `seq`: 전체 hourly bucket의 전역 순서 index입니다.
- `delta_t`: 동일 grid cell에서 이전 positive event 이후 경과한 hourly bucket 수입니다.
- `demand_qty`: 해당 grid cell/hour의 pickup count입니다.

In [ ]:
def yellow_trip_datetime_expr(df: pl.DataFrame) -> pl.Expr:
    """Parse pickup datetime robustly across local/server parquet schemas."""
    dtype = df.schema.get("tpep_pickup_datetime")
    expr = pl.col("tpep_pickup_datetime")
    if dtype == pl.Utf8:
        return expr.str.strptime(pl.Datetime, strict=False)
    return expr.cast(pl.Datetime, strict=False)


def build_yellow_trip_hourly_events(raw_df: pl.DataFrame) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Convert raw trip rows into positive-demand hourly grid-cell events."""
    required = {"tpep_pickup_datetime", "pickup_longitude", "pickup_latitude"}
    missing = sorted(required - set(raw_df.columns))
    if missing:
        raise ValueError(f"yellow_trip parquet is missing required columns: {missing}")

    working_df = raw_df
    if VENDOR_ID in (1, 2):
        working_df = working_df.filter(pl.col("VendorID") == int(VENDOR_ID))

    # Count positive pickups per spatial cell and hourly timestamp.
    events = (
        working_df.with_columns([
            yellow_trip_datetime_expr(working_df).alias("pickup_dt"),
            pl.col("pickup_longitude").cast(pl.Float64).alias("pickup_lon"),
            pl.col("pickup_latitude").cast(pl.Float64).alias("pickup_lat"),
        ])
        .filter(
            pl.col("pickup_dt").is_not_null()
            & pl.col("pickup_lon").is_not_null()
            & pl.col("pickup_lat").is_not_null()
            & pl.col("pickup_lon").is_between(MIN_LON, MAX_LON)
            & pl.col("pickup_lat").is_between(MIN_LAT, MAX_LAT)
        )
        .with_columns([
            (pl.col("pickup_lon") / GRID_SIZE_DEG).floor().cast(pl.Int64).alias("gx"),
            (pl.col("pickup_lat") / GRID_SIZE_DEG).floor().cast(pl.Int64).alias("gy"),
            pl.col("pickup_dt").dt.truncate(BUCKET_EVERY).alias("time_bucket"),
        ])
        .with_columns(
            (pl.col("gx").cast(pl.String) + pl.lit("_") + pl.col("gy").cast(pl.String)).alias("oper_part_no")
        )
        .group_by(["oper_part_no", "time_bucket"], maintain_order=True)
        .agg(pl.len().cast(pl.Float64).alias("demand_qty"))
    )

    # Global hourly index makes elapsed time comparable across grid cells.
    bucket_map = (
        events.select("time_bucket")
        .unique()
        .sort("time_bucket")
        .with_row_index("seq", offset=1)
        .with_columns(
            pl.col("time_bucket").dt.strftime("%Y%m%d%H%M%S").cast(pl.Int64).alias("demand_dt")
        )
    )

    events = (
        events.join(bucket_map, on="time_bucket", how="left")
        .select(["oper_part_no", "time_bucket", "demand_dt", "seq", "demand_qty"])
        .sort(["oper_part_no", "seq"])
    )

    series_stats = (
        events.group_by("oper_part_no")
        .agg([
            pl.len().alias("active_buckets"),
            pl.col("demand_qty").sum().alias("total_demand"),
        ])
        .sort(["active_buckets", "total_demand"], descending=[True, True])
    )

    if series_stats.height == 0:
        raise ValueError("No valid yellow-trip grid-cell events remained after filtering.")

    max_active = int(series_stats.select(pl.col("active_buckets").max()).item())
    effective_min_active = min(int(MIN_ACTIVE_BUCKETS), max_active)
    if effective_min_active < int(MIN_ACTIVE_BUCKETS):
        print(
            f"[WARN] requested MIN_ACTIVE_BUCKETS={MIN_ACTIVE_BUCKETS}, "
            f"but max active buckets={max_active}. Using {effective_min_active}."
        )

    eligible_series = series_stats.filter(pl.col("active_buckets") >= effective_min_active)
    if MAX_SERIES is not None:
        eligible_series = eligible_series.head(int(MAX_SERIES))

    event_df = (
        events.join(eligible_series.select("oper_part_no"), on="oper_part_no", how="inner")
        .sort(["oper_part_no", "seq"])
        .with_columns(
            (pl.col("seq") - pl.col("seq").shift(1).over("oper_part_no"))
            .fill_null(0)
            .cast(pl.Int32)
            .alias("delta_t")
        )
        .with_columns([
            pl.lit(RESOLUTION).alias("source_resolution"),
            pl.lit(float(GRID_SIZE_DEG)).alias("grid_size_deg"),
            pl.lit(int(effective_min_active)).alias("min_active_buckets"),
        ])
        .select([
            "oper_part_no",
            "demand_dt",
            "seq",
            "delta_t",
            "demand_qty",
            "time_bucket",
            "source_resolution",
            "grid_size_deg",
            "min_active_buckets",
        ])
    )

    return event_df, series_stats


yellow_hourly_df, yellow_series_stats = build_yellow_trip_hourly_events(raw_df)
print("hourly event rows =", yellow_hourly_df.height)
print("hourly series     =", yellow_hourly_df.select(pl.col("oper_part_no").n_unique()).item())
display(yellow_hourly_df.head(10))

## 5. 변환 결과 검증

학습 코드가 요구하는 핵심 컬럼이 모두 있는지 확인하고, series별 active bucket 분포와 quantity 분포를 간단히 확인합니다.

In [ ]:
required_output_cols = {"oper_part_no", "demand_dt", "seq", "delta_t", "demand_qty"}
missing_output_cols = sorted(required_output_cols - set(yellow_hourly_df.columns))
assert not missing_output_cols, f"Missing output columns: {missing_output_cols}"
assert yellow_hourly_df.filter(pl.col("demand_qty") <= 0).height == 0

summary = yellow_hourly_df.select([
    pl.len().alias("rows"),
    pl.col("oper_part_no").n_unique().alias("series"),
    pl.col("seq").min().alias("min_seq"),
    pl.col("seq").max().alias("max_seq"),
    pl.col("demand_qty").mean().alias("mean_qty"),
    pl.col("demand_qty").max().alias("max_qty"),
])
display(summary)

display(
    yellow_hourly_df.group_by("oper_part_no")
    .agg([
        pl.len().alias("events"),
        pl.col("demand_qty").sum().alias("total_qty"),
    ])
    .sort(["events", "total_qty"], descending=[True, True])
    .head(10)
)

## 6. Parquet 저장

아래 셀을 실행하면 학습 스크립트가 직접 읽을 preprocessed hourly event table이 생성됩니다.

저장 경로: `sample_data/new_york_taxi/yellow_trip_hourly.parquet`

In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
yellow_hourly_df.write_parquet(OUTPUT_PATH)
print(f"saved: {OUTPUT_PATH}")
print(f"rows={yellow_hourly_df.height:,}, series={yellow_hourly_df.select(pl.col('oper_part_no').n_unique()).item():,}")

## 7. 학습 코드에서 사용하는 방식

이제 long-epoch 실험에서는 raw `yellow_trip.parquet`이 아니라 아래처럼 preprocessed dataset을 선택합니다.

```bash
python ~/workspace/paper_research/simple_lab_test/search/tpp_experiment.py long-epoch \
  --datasets yellow_trip_hourly \
  --models rmtpp,titantpp,thp \
  --epochs 800 \
  --device cuda
```

이후 magnitude-factorized mark와 residual은 학습 코드에서 `demand_qty` 기준으로 다시 생성됩니다.